In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(msigdbr)
    library(rmcorr)
    library(ComplexHeatmap)
    library(circlize)
    library(viridis)
    library(glmmTMB)
    library(ggeffects)
    library(ggrepel)
})

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
    cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
    aggregate_marker_expression = counts(cds_gene_list_norm)
    aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
    aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
    aggregate_marker_expression = log(aggregate_marker_expression + 1)
    colData(cds)[[name]] = aggregate_marker_expression
    return(cds)
}

estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
    cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
    aggregate_g1s_expression = counts(cds_g1s)
    aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
    aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)

    cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
    aggregate_g2m_expression = counts(cds_g2m)
    aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
    aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
    cds$g1s_score = log(aggregate_g1s_expression+1)
    cds$g2m_score = log(aggregate_g2m_expression+1)
    cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
    return(cds)
}

normalize <- function(x) {
    return((x- min(x)) /(max(x)-min(x)))
}


In [ ]:
gene_set_BP = msigdbr::msigdbr(species = "mouse", subcategory = "GO:BP")
gene_set_CC = msigdbr::msigdbr(species = "mouse", subcategory = "GO:CC")
gene_set_hallmarks = msigdbr::msigdbr(species = "mouse", category = "H")
gene_set_reactome = msigdbr::msigdbr(species = "mouse", subcategory = "CP:REACTOME")
gene_set_cgp = msigdbr::msigdbr(species = "mouse", subcategory = "CGP")

gene_set_BP_df = gene_set_BP %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_hallmarks_df = gene_set_hallmarks %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_reactome_df = gene_set_reactome %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_CC_df = gene_set_CC %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

In [ ]:
granule_genes = gene_set_BP_df %>%
    filter(grepl("GOBP_STRESS_GRANULE_ASSEMBLY", gs_name)) %>%
    pull(gene_short_name)
upr_genes = gene_set_hallmarks_df %>%
    filter(grepl("UNFOLDED", gs_name)) %>%
    pull(gene_short_name)

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
cc.genes$s.genes = stringr::str_to_title(cc.genes$s.genes)
cc.genes$g2m.genes = stringr::str_to_title(cc.genes$g2m.genes)

## Estimate nucleolar score

In [ ]:
df_cell = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/E15.25-nucleolin-SiRDNA-SC35-combine-all-cds/df_cell.rds")
df_birth_series = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/JAX_data/df_cell.birth_series.rds")
df_early = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/JAX_data/df_cell.somitogenesis_validation.rds")

df_cell = bind_rows(df_cell, df_birth_series, df_early) %>%
             mutate(Size_Factor = UMI_count / exp(mean(log(UMI_count))))

dim(df_cell)

df_gene = read.csv("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/JAX_data/GSE228590_gene_annotation.csv")
head(df_gene)

In [ ]:
nuc_genesets = readRDS("../data/nucleolar_genes_Z5_0.7_0.4_n_6.RDS")
nuc_model = readRDS("../data/nucleolar_model_glmm_Z5_0.7_0.4_n_6.RDS")

In [ ]:
cds_list_names = c("JAX_E8-8.5.RDS", "JAX_E8-9.25.RDS", "JAX_E8.75-9.5.RDS", "JAX_E9.75-10-1.RDS", 
                   "JAX_E9.75-10-2.RDS", "JAX_E10.25-10.75.RDS", "JAX_E11-11.75.RDS", "JAX_E12-12.75.RDS", 
                   "JAX_E13-13.75.RDS", "JAX_E14-14.33.RDS", "JAX_E15-15.75.RDS", "JAX_E16-16.75.RDS", 
                   "JAX_E16-16.75-2.RDS", "JAX_E17-17.75.RDS", "JAX_E18-18.75.RDS", "JAX_E18-18.75-2.RDS", 
                   "JAX_P0-1.RDS", "JAX_P0-2.RDS", "JAX_birth-series.RDS")

df_all = NULL

for (cds_name in cds_list_names) {
    cat(cds_name, "...")
    
    # 1. Load CDS
    cds_path = paste0("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/JAX_data/cds_", cds_name)
    if(!file.exists(cds_path)) {
        cat(" File not found. Skipping.\n")
        next
    }
    cds_sub = readRDS(cds_path)
    
    df_sub_meta = df_cell[df_cell$cell_id %in% colData(cds_sub)$cell_id, ]
    colData(cds_sub)$Size_Factor = df_sub_meta$Size_Factor
    monocle3:::size_factors(cds_sub) = colData(cds_sub)$Size_Factor
    
    cds_sub = estimate_cell_cycle(cds_sub, 
                                  g1s_markers = cc.genes$s.genes, 
                                  g2m_markers = cc.genes$g2m.genes)
    
    cds_sub = aggregate_gene_score(cds_sub, granule_genes, "granule_score")
    cds_sub = aggregate_gene_score(cds_sub, upr_genes, "upr_score")
    

    genes_to_use <- nuc_genesets$gene_short_name
    score_name <- "nuc_score_Z5_0.7_0.4_n_6"
    cds_sub = aggregate_gene_score(cds_sub, genes_to_use, score_name)

    df_step = as.data.frame(colData(cds_sub))
    
    if (is.null(df_all)) {
        df_all = df_step
    } else {
        df_all = bind_rows(df_all, df_step)
    }
    
    cat(" Complete. Total rows in df_all:", nrow(df_all), "\n")
    rm(cds_sub, df_step)
    flush.console()
    gc()
}

head(df_all)

In [ ]:
s_name = "Z5_0.7_0.4_n_6"

raw_score_col <- paste0("nuc_score_", s_name)
norm_score_col <- paste0("normalized_nuc_score_", s_name)
pred_col <- paste0("score_pred_glmm_", s_name)


df_all[[norm_score_col]] <- normalize(df_all[[raw_score_col]])
target_var <- all.vars(formula(nuc_model))[2] 

pred_df <- data.frame(df_all[[norm_score_col]])
colnames(pred_df) <- target_var

df_all[[pred_col]] <- as.double(predict(nuc_model, 
                                        newdata = pred_df, 
                                        re.form = NA, 
                                        type = "response"))
        

In [ ]:
df_all = df_all %>%
    mutate(day = ifelse(day == "E8.0-E8.5", "E8.5", day)) %>%
    mutate(day_numeric = as.numeric(sub('.', '', day))) %>%
    mutate(day_numeric = ifelse(day_numeric == 0, 19, day_numeric)) 

In [ ]:
df_embryo = df_all %>% filter(!experimental_id %in% c("run_28", "run_29"))
df_birth = df_all %>% filter(experimental_id == "run_28")

dim(df_embryo)
dim(df_birth)

In [ ]:
df_embryo[[norm_score_col]] <- normalize(df_embryo[[raw_score_col]])
target_var <- all.vars(formula(nuc_model))[2] 

pred_df <- data.frame(df_embryo[[norm_score_col]])
colnames(pred_df) <- target_var

df_embryo[[pred_col]] <- as.double(predict(nuc_model, 
                                        newdata = pred_df, 
                                        re.form = NA, 
                                        type = "response"))
        

In [ ]:
df_birth[[norm_score_col]] <- normalize(df_birth[[raw_score_col]])
target_var <- all.vars(formula(nuc_model))[2] 

pred_df <- data.frame(df_birth[[norm_score_col]])
colnames(pred_df) <- target_var

df_birth[[pred_col]] <- as.double(predict(nuc_model, 
                                        newdata = pred_df, 
                                        re.form = NA, 
                                        type = "response"))
        

In [ ]:
saveRDS(df_all, "df_all_fraction_glmm_stouffers_embryo.RDS")

In [ ]:
saveRDS(df_embryo, "df_embryo_fraction_glmm_stouffers_embryo.RDS")

In [ ]:
saveRDS(df_birth, "df_birth_fraction_glmm_stouffers_embryo.RDS")